In [1]:
from ccdc import io
from ccdc.io import EntryReader

from collections import defaultdict
from itertools import islice

from scipy.constants import year
from tqdm import tqdm

import pandas as pd

import time

from concurrent.futures import ProcessPoolExecutor

In [ ]:
ENTRY_PROPS = [
    "identifier",
    "is_organic",
    "has_disorder",
    "disorder_details",
    "color",
    "polymorph",
    "phase_transition",
    "radiation_source",
    "temperature",
    "calculated_density",
    "melting_point",
    "melting_point_default_units",
#    "heat_capacity",        # Solubility Platform not available
#    "heat_capacity_notes",
#    "heat_of_fusion",
#    "heat_of_fusion_notes",
#    "solubility_data",
    "bioactivity",
    "ccdc_number",
    "doi",
#   "publications",         # Citation tuple → extracted manually in chunk_extraction
    "solvent",
]

CRYSTAL_PROPS = [
#   "spacegroup_number_and_setting",  # (number, setting) tuple → split into 2 columns in chunk_extraction
    "spacegroup_symbol",
#   "symmetry_operators",             # str tuple → extracted manually in chunk_extraction
    "z_prime",
    "z_value",
    "cell_volume",
#   "cell_lengths",                   # non-native object → split into 3 columns in chunk_extraction
#   "cell_angles",                    # non-native object → split into 3 columns in chunk_extraction
]

MOLECULE_PROPS = [
    "smiles",
    "molecular_volume",
    "molecular_weight",
]


In [3]:
csd = EntryReader('CSD')
csd_reader = io.EntryReader('CSD')

### 1. Extract all records to single CSV table

In [ ]:
all_df = []

start = time.time()

total_entry = 500
for entry in tqdm(islice(csd, total_entry), total=total_entry, desc="Processing"):

    # check organic
    is_organic = entry.is_organic

    # check num components
    num_component = len(entry.molecule.components)

    # check metal
    has_metal = any(atom.is_metal for atom in entry.molecule.atoms)

    # add result
    all_df.append({
        "ID":entry.identifier,
        "SMILES": entry.molecule.smiles,
        "IS_ORGANIC": is_organic,
        "HAS_METAL":has_metal,
        "NUM_COMPONENT": num_component,
    })

delta_time = round(time.time() - start, 1)

all_df = pd.DataFrame(all_df)
all_df.to_csv("csd_all.csv", index=False)

print(f"Sequential : Processed {len(all_df)//1000}k entries in {delta_time}s")

In [11]:
all_df

,ID,SMILES,IS_ORGANIC,HAS_METAL,NUM_COMPONENT
0,AABHTZ,CC(=O)NN1C=NN=C1N(N=Cc1c(Cl)cccc1Cl)C(C)=O,True,False,1
1,AACANI10,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
2,AACANI11,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
3,AACFAZ,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
4,AACFAZ10,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
...,...,...,...,...,...
495,ABAZPT,CC(=O)N1=NC(=[NH][Pt]21[NH]=C(N=N2C(C)=O)C(C)(...,False,True,1
496,ABAZUA,Cc1cc(C)c(c(C)c1)B1(c2ccccc2c2ccc(cn12)c1ccc(c...,True,False,1
497,ABAZUB,O.COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23...,True,False,2
498,ABAZUB01,COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23)C...,True,False,2


In [ ]:
NATIVE_TYPES = (bool, int, float, str, type(None))

COLUMN_ORDER = [
    # ENTRY
    "identifier", "is_organic", "has_metal", "num_components",
    "has_disorder", "disorder_details", "color",
    "polymorph", "phase_transition", "radiation_source", "temperature",
    "calculated_density", "melting_point", "melting_point_default_units",
    "bioactivity", "ccdc_number", "doi", "publications", "publication_years", "solvent",
    # MOLECULE
    "smiles", "molecular_volume", "molecular_weight", "inchi",
    # CRYSTAL
    "spacegroup_number", "spacegroup_setting", "spacegroup_symbol",
    "symmetry_operators", "z_prime", "z_value",
    "cell_length_a", "cell_length_b", "cell_length_c",
    "cell_angle_alpha", "cell_angle_beta", "cell_angle_gamma",
    "cell_volume",
]

start = time.time()

def chunk_extraction(chunk_range):
    start, stop = chunk_range
    results = []
    warnings = set()
    with io.EntryReader('CSD') as reader:
        for i in range(start, stop):
            entry = reader[i]
            mol = entry.molecule
            crys = entry.crystal

            row = {}
            for prop in ENTRY_PROPS:
                try:
                    val = getattr(entry, prop, None)
                    if not isinstance(val, NATIVE_TYPES):
                        warnings.add(f"[ENTRY] {prop} -> {type(val).__name__} : {repr(val)[:60]}")
                        val = None
                    row[prop] = val
                except Exception:
                    row[prop] = None

            for prop in MOLECULE_PROPS:
                try:
                    val = getattr(mol, prop, None)
                    if not isinstance(val, NATIVE_TYPES):
                        warnings.add(f"[MOLECULE] {prop} -> {type(val).__name__} : {repr(val)[:60]}")
                        val = None
                    row[prop] = val
                except Exception:
                    row[prop] = None

            for prop in CRYSTAL_PROPS:
                try:
                    val = getattr(crys, prop, None)
                    if not isinstance(val, NATIVE_TYPES):
                        warnings.add(f"[CRYSTAL] {prop} -> {type(val).__name__} : {repr(val)[:60]}")
                        val = None
                    row[prop] = val
                except Exception:
                    row[prop] = None

            try:
                sg = crys.spacegroup_number_and_setting
                row["spacegroup_number"] = sg[0] if sg else None
                row["spacegroup_setting"] = sg[1] if sg else None
            except Exception:
                row["spacegroup_number"] = None
                row["spacegroup_setting"] = None

            try:
                pubs = entry.publications
                row["publications"] = "|".join(str(p) for p in pubs)
                years = [str(p.year) for p in pubs if p.year is not None and p.year >= 1800]
                row["publication_years"] = "|".join(years) if years else None
            except Exception:
                row["publications"] = None
                row["publication_years"] = None

            try:
                row["symmetry_operators"] = "|".join(crys.symmetry_operators)
            except Exception:
                row["symmetry_operators"] = None

            try:
                row["inchi"] = mol.generate_inchi().inchi
            except Exception:
                row["inchi"] = None

            try:
                cl = crys.cell_lengths
                row["cell_length_a"] = cl.a
                row["cell_length_b"] = cl.b
                row["cell_length_c"] = cl.c
            except Exception:
                row["cell_length_a"] = None
                row["cell_length_b"] = None
                row["cell_length_c"] = None

            try:
                ca = crys.cell_angles
                row["cell_angle_alpha"] = ca.alpha
                row["cell_angle_beta"] = ca.beta
                row["cell_angle_gamma"] = ca.gamma
            except Exception:
                row["cell_angle_alpha"] = None
                row["cell_angle_beta"] = None
                row["cell_angle_gamma"] = None

            row["has_metal"] = any(atom.is_metal for atom in mol.atoms)
            row["num_components"] = len(mol.components)

            results.append(row)
    return results, warnings

with io.EntryReader('CSD') as reader:
    NUMBER_ENTRIES = len(reader)

# NUMBER_ENTRIES = 50000
NUM_WORKERS = 24
CHUNK_SIZE = 500
chunk_ranges = [(i, min(i + CHUNK_SIZE, NUMBER_ENTRIES)) for i in range(0, NUMBER_ENTRIES, CHUNK_SIZE)]
pbar = tqdm(total=NUMBER_ENTRIES, unit="entry", desc=f"Extraction ({NUM_WORKERS} workers)", smoothing=0.1)

all_results = []
all_warnings = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    for chunk_results, chunk_warnings in executor.map(chunk_extraction, chunk_ranges):
        all_results.extend(chunk_results)
        all_warnings.update(chunk_warnings)
        pbar.update(len(chunk_results))
pbar.close()

delta_time = round(time.time() - start, 1)

all_df = pd.DataFrame(all_results)[COLUMN_ORDER]
all_df.to_csv("csd_all.csv", index=False)

print(f"Processed {len(all_df)//1000}k entries in {delta_time}s ({NUM_WORKERS} workers)")

if all_warnings:
    print("\n⚠️  Non-native types detected (set to None):")
    for w in sorted(all_warnings):
        print(" ", w)


In [ ]:
all_df

### 2. Select only organic one component molecules

In [ ]:
all_df = pd.read_csv("csd_all.csv")

df_filtered = all_df[
    (all_df["is_organic"] == True) &
    (all_df["has_metal"] == False) &
    (all_df["num_components"] == 1)
]

print(f"Selected {len(df_filtered)} entries")

In [ ]:
df_filtered

### 3. Count the number of crystal forms

In [91]:
df_with_inchi = df_filtered.dropna(subset=["inchi"])
groups = df_with_inchi.groupby("inchi")["identifier"].apply(list).to_dict()

print(f"Selected {len(groups)} molecules with InChI")

Selected 324985 molecules with InChI


In [100]:
id_to_smiles = df_with_inchi.set_index("identifier")["smiles"]
id_to_dates = df_with_inchi.set_index("identifier")["publication_years"]

df_counted = []
for key, entries in groups.items():
    years = [str(id_to_dates[entry]) for entry in entries]
    df_counted.append({
        "smiles": id_to_smiles[entries[0]],
        "num_forms": len(entries),
        "publication_years": "|".join(years) if years else None,
    })

df_counted = pd.DataFrame(df_counted)
df_counted = df_counted.drop_duplicates("smiles")

print(f"Selected {len(df_counted)} molecules with counted forms")

Selected 316081 molecules with counted forms


In [101]:
df_counted

,smiles,num_forms,publication_years
0,[Se-][As]1[As]2[As]3[Se][As]4[As]5[As]([Se-])[...,1,1991.0
1,ClB123[As]45[As]61B14(Cl)B25(Cl)B361Cl,1,1995.0
2,IB1234[BH]567[BH]89%10[BH]%11%12%13[BH]158B12%...,1,1991.0
3,Cl[As]12(Cl)[O-][As]3(Cl)(Cl)Cl41[As]1(Cl)(Cl)...,1,2001.0
4,S1[As]2S[As]3[As]1[As]3S2,1,2019.0
...,...,...,...
324979,S=P12SP3(=S)SP(=S)(S1)SP(=S)(S2)S3,2,1998.0|2025.0
324980,S1P2SP3P1P3S2,1,1997.0
324981,S=P12SP3SP(S1)P2S3,1,2025.0
324983,S=P12SP3SP(=S)(SP3S1)S2,1,2025.0


In [102]:
n_total = len(df_counted)
n_mono = sum(df_counted["num_forms"] == 1)
n_mono_db = df_with_inchi["polymorph"].isna().sum()
n_poly = sum(df_counted["num_forms"] > 1)

print(f"Total molecules: {n_total}")
print(f"Monomorphs: {n_mono} ({round(100 * n_mono / n_total, 1)} %)")
print(f"Monomorphs registered in db: {n_mono_db} ({round(100 * n_mono_db / len(df_with_inchi), 1)} %)")
print(f"Polymorphs: {n_poly} ({round(100 * n_poly / n_total, 1)} %)")


Total molecules: 316081
Monomorphs: 300936 (95.2 %)
Monomorphs registered in db: 339929 (96.1 %)
Polymorphs: 15145 (4.8 %)


In [ ]:
mono_db_inchi = set(df_with_inchi[df_with_inchi["polymorph"].isna()]["inchi"])
poly_inchi = {inchi for inchi, entries in groups.items() if len(entries) > 1}
print(len(poly_inchi))

false_monomorphs = mono_db_inchi & poly_inchi
print(f"Molecules marked as monomorphs in DB but polymorphic by InChI: {len(false_monomorphs)}")
print(f"i.e. {round(100 * len(false_monomorphs) / len(mono_db_inchi), 1)}% of DB monomorphs")

In [104]:
df_counted.to_csv("csd_counted.csv", index=False)

### 4. Properties

In [9]:
entry = csd[0]
properties = dir(entry.crystal)
properties

['Contact',
 'Disorder',
 'HBond',
 'MillerIndices',
 'PeriodicBondChain',
 'ReducedCell',
 'Void',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_chemical_info',
 '_crystal',
 '_crystal_info',
 '_csv',
 '_decode_inclusion',
 '_identifier',
 '_voids_volume_telemetry',
 'add_hydrogens',
 'are_atoms_symmetry_related',
 'assign_bonds',
 'asymmetric_unit_molecule',
 'atoms_on_special_positions',
 'calculate_voids',
 'calculated_density',
 'cell_angles',
 'cell_lengths',
 'cell_volume',
 'centre_molecule',
 'contact_network',
 'contacts',
 'copy',
 'crystal_system',
 'disorder',
 'disordered_molecule',
 'formula',
 'fractional_to_orthogonal',
 'from_stri

In [ ]:
BANNED_LIST = ['melting_point_display_string', 'input_melting_point_range', 'formatted_melting_point_text', 'formatted_melting_point_range' ]
MAX_ENTRIES_TEST = 100

prop_list = {}
for prop in dir(entry):
    if prop.startswith('_') or prop in BANNED_LIST:
        continue

    try:
        attr_class = getattr(type(entry), prop, None)
        if isinstance(attr_class, property):
            doc = getattr(attr_class.fget, '__doc__', None)
        else:
            doc = getattr(getattr(entry, prop), '__doc__', None)

        doc_clean = doc.split('>>>')[0].split(':')[0].strip() if doc else None

        example_value = None
        for i, test_entry in enumerate(islice(csd, MAX_ENTRIES_TEST)):
            try:
                value = getattr(test_entry, prop)
                if value:
                    example_value = value
                    break
            except:
                continue

        # if str(example_value)[0] == '<':
        #     continue

        prop_list[prop] = {
            "description": doc_clean,
            "example": str(example_value) if example_value is not None else "No example found"
        }

    except Exception as e:
        if "Solubility Platform" not in str(e):
            print(f"Error with {prop} : {e}")

for prop, infos in prop_list.items():
    if infos['example'][0] == "<":
        print(f"--- {prop} ---")
        print(f"Description : {infos['description']}")
        print(f"Exemple : {infos['example']}")
        print()

print(len(prop_list))

In [ ]:
BANNED_LIST = ['melting_point_display_string', 'input_melting_point_range',
               'formatted_melting_point_text', 'formatted_melting_point_range']
MAX_ENTRIES_TEST = 100

entry = csd[0]
molecule = entry.molecule

prop_list = {}
for prop in dir(molecule):
    if prop.startswith('_') or prop in BANNED_LIST:
        continue

    try:
        attr_class = getattr(type(molecule), prop, None)
        if isinstance(attr_class, property):
            doc = getattr(attr_class.fget, '__doc__', None)
        else:
            doc = getattr(getattr(molecule, prop), '__doc__', None)

        doc_clean = doc.split('>>>')[0].split(':')[0].strip() if doc else None

        example_value = None
        for test_entry in islice(csd, MAX_ENTRIES_TEST):
            try:
                value = getattr(test_entry.molecule, prop)
                if value:
                    example_value = value
                    break
            except:
                continue

        # if str(example_value)[0] == '<':
        #     continue
        
        prop_list[prop] = {
            "description": doc_clean,
            "example": str(example_value) if example_value is not None else "No example found"
        }

    except Exception as e:
        if "Solubility Platform" not in str(e):
            print(f"Error with {prop} : {e}")

for prop, infos in prop_list.items():
    print(f"--- {prop} ---")
    print(f"Description : {infos['description']}")
    print(f"Exemple : {infos['example']}")
    print()

print(len(prop_list))